# name: seba saud

# Task 1: Text Generation Temperature

In [15]:
from langchain_openai import ChatOpenAI


api_key = "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"
base_url = "https://openrouter.ai/api/v1"

low_temp_model = ChatOpenAI(api_key=api_key, base_url=base_url, model="nvidia/nemotron-3-nano-30b-a3b:free", temperature=0)

high_temp_model = ChatOpenAI(api_key=api_key, base_url=base_url, model="nvidia/nemotron-3-nano-30b-a3b:free", temperature=1.5)

prompt = "Explain the moon in one sentence. Make it poetic and format as JSON."

print("Low Temperature Result ")
low_res = low_temp_model.invoke(prompt)
print(low_res.content)

print("\nHigh Temperature Result ")
high_res = high_temp_model.invoke(prompt)
print(high_res.content)

Low Temperature Result 
{
  "explanation": "She glows, a silver whisper stitching night's tapestry with dreams of forgotten tides."
}

High Temperature Result 
{
  "sentence": "The moon, a silver harp strummed by winter's sighs, glows like a quiet hymn that stitches the dark night wish."
}


# Task 2: Sentiment Analysis

In [10]:
from typing import Literal


class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]

structured_llm_sentiment = low_temp_model.with_structured_output(SentimentAnalysis)


sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily."
]

for text in sentences:
    result = structured_llm_sentiment.invoke(text)
    print(f"Text: {text} -> Sentiment: {result.sentiment}")

Text: Kindness creates lasting joy. -> Sentiment: positive
Text: Success rewards persistent effort. -> Sentiment: positive
Text: I love Sunlight. It warms the skin. -> Sentiment: positive
Text: Pessemestic all the time. -> Sentiment: positive
Text: The storm caused damage! -> Sentiment: neutral
Text: The clock ticks steadily. -> Sentiment: neutral


# Task 3: Categorization

In [14]:


class Categorization(BaseModel):
    
    tags: List[Literal["cars", "shopping", "sports", "study", "work"]]

structured_llm_tags = low_temp_model.with_structured_output(Categorization)

texts = [
    "That restoration looks incredible; you have a real talent for mechanics.",
    "I found the perfect gift today! The staff was incredibly helpful.",
    "Great game today! Your teamwork was the key to that victory.",
    "Learning together helped me finally grasp these concepts. Thank you!"
]

print(" Categorization Results:")
for t in texts:
   
    result = structured_llm_tags.invoke(f"Categorize this text using only the allowed tags: {t}")
    print(f"Text: {t} -> Tags: {result.tags}")

 Categorization Results:
Text: That restoration looks incredible; you have a real talent for mechanics. -> Tags: ['cars']
Text: I found the perfect gift today! The staff was incredibly helpful. -> Tags: ['cars']
Text: Great game today! Your teamwork was the key to that victory. -> Tags: ['sports']
Text: Learning together helped me finally grasp these concepts. Thank you! -> Tags: ['cars', 'sports']


In [7]:
%pip install langchain-community pypdf

   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
   ---------------------------- ----------- 1.8/2.5 MB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 2.5/2.5 MB 6.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 7.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Task 4: Data Extraction

In [8]:
from langchain_community.document_loaders import PyPDFLoader

class ResumeData(BaseModel):
    candidate_name: str = Field(description="The full name of the candidate")
    skills: List[str] = Field(description="A list of technical and soft skills")
    experience_years: int = Field(description="Total number of years of experience as an integer")
    education: str = Field(description="Highest degree obtained")

file_path = "my_cv.pdf" 
loader = PyPDFLoader(file_path)
pages = loader.load()
full_text = " ".join([page.page_content for page in pages])

structured_extractor = low_temp_model.with_structured_output(ResumeData)

print("--- Extracting Data from CV ---")
extracted_info = structured_extractor.invoke(f"Extract key information from this CV text: {full_text}")

print(f"Name: {extracted_info.candidate_name}")
print(f"Skills: {extracted_info.skills}")
print(f"Experience: {extracted_info.experience_years} years")

--- Extracting Data from CV ---
Name: Seba Saud
Skills: ['Problem Solving', 'Critical & Analytical Thinking', 'Teamwork & Collaboration', 'Leadership', 'Communication Skills', 'Time Management', 'Adaptability', 'Fast Learner', 'Creativity', 'Initiative', 'Responsibility', 'Attention to Detail', 'Stress Management']
Experience: 0 years


# Task 5: Tools (Function Calling)

In [9]:
from typing import Any, Dict

class ToolCall(BaseModel):
    tool_name: str = Field(description="The name of the function to use (add, subtract, multiply, divide)")
    arguments: Dict[str, Any] = Field(description="The parameters to pass, e.g., {'a': 5, 'b': 10}")


system_prompt = """
You are a math assistant. You have access to these tools:
- add(a, b): Adds two numbers.
- subtract(a, b): Subtracts b from a.
- multiply(a, b): Multiplies two numbers.
- divide(a, b): Divides a by b.

Return the tool name and arguments required to solve the user's request in a structured format.
"""

structured_tool_llm = low_temp_model.with_structured_output(ToolCall)

user_query = "What is 15 multiplied by 4?"
response = structured_tool_llm.invoke(f"{system_prompt}\n\nUser Question: {user_query}")

print(f"Decision: The model decided to use the tool: '{response.tool_name}'")
print(f"Arguments: {response.arguments}")

Decision: The model decided to use the tool: 'multiply'
Arguments: {'a': 15, 'b': 4}
